# Alignment Score Properties

This Jupyter notebook is intended to support the Appendix Alignment Score Properties. We
explore various implications and properties of alignment and make plots. 

## Basin of Attraction

We argue that the largest basin of attraction is given by a highly aligned matrix. To do that,
we generate various matrices with various amounts of alignment and test the proportion of 
initial expression states that develop into the target phenotype as its stable state.

In [ ]:
using LinearAlgebra, Distributions, Random, Plots, StatsBase, LaTeXStrings, ProgressMeter
using Base.Threads

include("../src/wagner_algorithm.jl")
using ..BooleanNetwork: develop

include("../src/data_processing.jl")
using ..CustomStats: alignment_score, compute_mut_robustness

Plots.theme(:dao)

In [ ]:
function create_state_space(N)
    vectors = [collect(v) for v in Iterators.product(fill([-1.0, 1.0], N)...)]
    return vectors[:]
end

function create_signs_aligned_matrix(number_genes, target_vector)
    sign_mat = [target_vector[i] * target_vector[j] for i in 1:number_genes, j in 1:number_genes]
    return sign_mat
end

function create_matrix_alignment(weight_dist, number_genes, number_aligned_weights, target_vector)
    sign_matrix = create_signs_aligned_matrix(number_genes, target_vector)
    
    cartesian_indices = sample(CartesianIndices(sign_matrix), 
        number_genes^2-number_aligned_weights; 
        replace=false)

    for cartesian_idx in cartesian_indices
        sign_matrix[cartesian_idx] = sign_matrix[cartesian_idx] * -1.0
    end

    weight_matrix = abs.(rand(weight_dist, (number_genes,number_genes)))
    return sign_matrix .* weight_matrix
end

function compute_basin_size(state_space, matrix, target_vector)
    # generate all expressions from the state space
    phenotype_expression = develop.(Ref(matrix), state_space, Ref(100), Ref(sign), Ref(:synchronous))
    # Count the number of states in which the stable state was the target vector
    
    phenotype_vectors = [phen[1] for phen in phenotype_expression]
    matches = findall(x -> x == target_vector, phenotype_vectors)
    return length(matches) / length(state_space)
end

In [ ]:
number_genes = 14
replicates = 20
weight_dist = Normal(0.0, 1.0)
state_space = create_state_space(number_genes)
target_vector = rand([-1.0, 1.0], number_genes)

alignment_scores = Matrix{Float64}(undef, replicates, number_genes^2+1)
basin_sizes = Matrix{Float64}(undef, replicates, number_genes^2+1)


progress = Progress(replicates; desc="Replicates")
@threads for rep_idx in 1:replicates
    for alig_num in 0:number_genes^2
        matrix = create_matrix_alignment(weight_dist, number_genes, alig_num, target_vector)
        basin_size = compute_basin_size(state_space, matrix, target_vector)
        alignment_score_indiv = alignment_score(matrix, target_vector)

        alignment_scores[rep_idx, alig_num+1] = alignment_score_indiv
        basin_sizes[rep_idx, alig_num+1] = basin_size
    end
    next!(progress)
end

In [ ]:
p1 = scatter(
    vec(alignment_scores), 
    vec(basin_sizes),
    xlabel = "Alignment Score " * L"A(W)",
    ylabel = "Probability of converging to " * L"P",
    alpha = 0.4, 
    markersize = 4,
    label = "",
    color =:black, 
    grid=false,
    majorgrid=false,
    minorgrid=false,
)

In [ ]:
savefig(p1, "basin_size.png")

## Alignment increases mutational robustness

We argue that a highly aligned matrix has mutational robustness metrics that indicate a
narrower expressed phenotype distribution. We generate 200 matrices to different amounts of
aligned weights and generate the mutational robustness metrics of $\Delta \bar{v}$, $\sigma^2$,
$\Delta p_u$, $\rho^2$. We plot them together for comparison.

In [ ]:
number_genes = 10
replicates = 200
weights_dist = Normal(0.0, 1.0)
target_vector = rand([-1.0, 1.0], number_genes)
initial_state = rand([-1.0, 1.0], number_genes)
alignment_values = [Int(number_genes^2 / 2), Int(3 * number_genes^2 / 4), Int(number_genes^2)]

n_mutations = 30
n_noise_masks = 100
noise_dist = Gamma(1.0,1.0)
mutation_prob = 0.01

shift = Matrix{Float64}(undef, 3, replicates)
var = Matrix{Float64}(undef, 3, replicates)
prob_shift = Matrix{Float64}(undef, 3, replicates)
prob_var = Matrix{Float64}(undef, 3, replicates)

progress = Progress(replicates; desc="Replicates")
@threads for rep_idx in 1:replicates
    for (alig_idx, alig_num) in enumerate(alignment_values)
        matrix = create_matrix_alignment(weight_dist, number_genes, alig_num, target_vector)

        result = compute_mut_robustness(
        matrix, initial_state, n_mutations, n_noise_masks, noise_dist, mutation_prob;
        weights_dist=weights_dist, max_steps=100,
        activation_function=sign,
        development_type=:asynchronous
        )

        shift[alig_idx, rep_idx] = result.stable_expression_shift
        var[alig_idx, rep_idx] = result.stable_expression_var
        prob_shift[alig_idx, rep_idx] = result.unstable_prob_shift
        prob_var[alig_idx, rep_idx] = result.unstable_prob_var
    end
end

In [ ]:
function plot_histogram_metric(metric_matrix; 
    bins=30, title="", colors=["lightsalmon", "coral2", "firebrick2"],
    xlabel ="", ylabel="", labels = ["y1", "y2", "y3"])

    p1 = histogram(metric_matrix[1,:], bins=bins, title=title, color=colors[1], alpha = 0.4,
        xlabel = xlabel, ylabel=ylabel, label=labels[1], majorgrid=false, minorgrid=false, grid=false,
        linecolor=:black, lw=0.5)
    histogram!(p1, metric_matrix[2, :], bins=bins, color=colors[2], alpha=0.4, label=labels[2],
        linecolor=:black, lw=0.5)
    histogram!(p1, metric_matrix[3, :], bins=bins, color=colors[3], alpha=0.8, label=labels[3],
        linecolor=:black, lw=0.5, legend=:best)

    return p1
end;

In [ ]:
pink_steps = range(colorant"#F8CAE4", stop=colorant"#832D51", length=6)

In [ ]:
green_steps = range(colorant"#CFDD9D", stop=colorant"#447A5F", length=6)

In [ ]:
colors_top_1 = pink_steps[4:6]
colors_top_2 = pink_steps[1:3]
colors_bot_1 = green_steps[4:6]
colors_bot_2 = green_steps[1:3]

shift_hist = plot_histogram_metric(
    shift, bins=range(minimum(shift), maximum(shift)*1.03, length=40),
    title="Stable expression shift", colors=colors_top_1, #[:pink, :hotpink2, :maroon3],
    xlabel=L"\Delta \bar{v}",
    ylabel="Frequency",
    labels=["$val / $(number_genes^2) aligned weights" for val in alignment_values]
)

expr_var_hist = plot_histogram_metric(
    var, bins=range(minimum(var), maximum(var)*1.03, length=40),
    title="Stable expression variance", colors=colors_top_2, #[:mediumorchid1, :mediumpurple2, :blueviolet],
    xlabel=L"\sigma^2",
    ylabel="Frequency",
    labels=["$val / $(number_genes^2) aligned weights" for val in alignment_values]
)
prob_shift_hist = plot_histogram_metric(
    prob_shift, bins=range(minimum(prob_shift), maximum(prob_shift)*1.03, length=40),
    title="Instability shift", colors=colors_bot_1, #[:deepskyblue, :dodgerblue2, :royalblue4],
    xlabel=L"\Delta p_u",
    ylabel="Frequency",
    labels=["$val / $(number_genes^2) aligned weights" for val in alignment_values]
)

prob_var_hist = plot_histogram_metric(
    prob_var, bins=range(minimum(prob_var), maximum(prob_var)*1.03, length=40),
    title="Instability variance", colors=colors_bot_2, #[:paleturquoise1, :turquoise1, :teal],
    xlabel=L"\rho^2",
    ylabel="Frequency",
    labels=["$val / $(number_genes^2) aligned weights" for val in alignment_values]
)

histograms = [shift_hist, expr_var_hist, prob_shift_hist, prob_var_hist]

combined_plot = plot(histograms..., layout = (2,2), size=(900,700))

savefig(combined_plot, "histograms_mut_alignment.png")
display(combined_plot)